## **SETUP ENV**


In [ ]:
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ind
!pip install prometheus-eval pdf2image pytesseract PyPDF2 accelerate bitsandbytes
!pip install PyMuPDF

## **ENG**

In [ ]:
## versi single high GPU 
import json
import pandas as pd
import pytesseract
import torch
import gc
import os
import re
import glob
import csv
import fitz
from pdf2image import convert_from_path
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

# =========================================================
# 1. KONFIGURASI & SETUP
# =========================
JSON_DIR = '/content/drive/MyDrive/KP/dataset/D12-Clinical Handbook Psychological Disorders'
PDF_DIR = '/content/drive/MyDrive/KP/dokumen/D12-Clinical Handbook Psychological Disorders'
OUTPUT_BASE_PATH = '/content/drive/MyDrive/KP/output-D12-H-1'
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)
folder_name = os.path.basename(JSON_DIR)
output_csv = os.path.join(OUTPUT_BASE_PATH, f"{folder_name}_results3.csv")

# =========================================================
# 2. MODELS LOADING 
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using Device: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

print("📦 Loading NLLB (Translation) to GPU...")
trans_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
trans_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    torch_dtype=torch.float16,
    use_safetensors=True
).to(device)

print("🤖 Loading Prometheus (Judge) in 4-bit...")
eval_tokenizer = AutoTokenizer.from_pretrained("prometheus-eval/prometheus-7b-v2.0")
eval_model = AutoModelForCausalLM.from_pretrained(
    "prometheus-eval/prometheus-7b-v2.0",
    device_map="auto",
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16, 
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
)

# =========================================================
# 3. HELPERS
# =========================================================
def extract_page_range(filename):
    match = re.search(r'(\d+)-(\d+)', filename)
    return f"{match.group(1)}-{match.group(2)}" if match else None

def convert_score(score):
    return (score - 1) / 4.0

def translate_text(text):
    if not text or text.strip() == "": return ""
    trans_tokenizer.src_lang = "ind_Latn"
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        res = trans_model.generate(
            **inputs,
            forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=512
        )
    return trans_tokenizer.decode(res[0], skip_special_tokens=True)

def hf_judge(prompt):
    safe_limit = 3584
    inputs = eval_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=safe_limit
    ).to(eval_model.device)

    with torch.no_grad():
        out = eval_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=eval_tokenizer.eos_token_id
        )

    decoded = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    match = re.search(r'\[RESULT\]\s*(\d)', decoded)
    score = int(match.group(1)) if match else 1
    feedback = decoded.split("[RESULT]")[0].replace("[FEEDBACK]", "").strip()
    return feedback, score

# =========================================================
# 4. RUBRIC SETUP
# =========================
rubric_data = {
    "criteria": "Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context.",
    "score1_description": "The answer contains severe hallucinations with information that completely contradicts or is entirely absent from the document context.",
    "score2_description": "The answer contains significant hallucinations with multiple pieces of information that are inconsistent or add details not present in the context.",
    "score3_description": "The answer contains moderate hallucinations with some information that doesn't align with the context or includes inappropriate generalizations.",
    "score4_description": "The answer is mostly consistent with the context but contains minor hallucinations or slightly inaccurate interpretations.",
    "score5_description": "The answer is completely faithful to the context with no hallucinations, and all information is accurately derived from the document."
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# =========================================================
# 5. RESUME PREPARATION
# =========================
processed_pairs = set()
if os.path.exists(output_csv):
    try:
        df_existing = pd.read_csv(output_csv)
        # Menggunakan kombinasi file sumber dan pertanyaan original sebagai unik ID
        for _, row in df_existing.iterrows():
            processed_pairs.add((str(row['source_file']), str(row['question_id_original'])))
        print(f"🔄 RESUME: Berhasil memuat {len(processed_pairs)} soal yang sudah selesai.")
    except Exception as e:
        print(f"⚠️ Gagal memuat file lama, memulai dari nol: {e}")

# Buat Header jika file baru
if not os.path.exists(output_csv):
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'folder', 'page_range', 'question_id', 'question_id_original',
            'question_en', 'answer_id_original', 'answer_en',
            'score_5', 'score_01', 'feedback', 'is_hallucinated', 'source_file'
        ])

# =========================================================
# 6. MAIN PROCESSING LOOP
# =========================================================
json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))

for j_path in json_files:
    f_name = os.path.basename(j_path)
    p_range = extract_page_range(f_name)

    with open(j_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)

    pending_tasks = []
    for i, item in enumerate(qna_data):
        if (f_name, str(item['Question'])) not in processed_pairs:
            pending_tasks.append((i, item))

    if not pending_tasks:
        print(f"⏭️  File {f_name} sudah selesai semua. Skip.")
        continue

    # Cari PDF pasangannya
    matching_pdf = next((p for p in pdf_files if extract_page_range(os.path.basename(p)) == p_range), None)
    if not matching_pdf:
        print(f"⚠️ PDF tidak ditemukan untuk: {f_name}")
        continue

    print(f"\n--- 📖 Memproses {len(pending_tasks)} soal tersisa di: {f_name} ---")

    # Ekstraksi Teks (PyMuPDF)
    try:
        # doc = fitz.open(matching_pdf)
        # context_text = "".join([page.get_text("text", sort=True) for page in doc]) >> kalau misal dokumennya 2 kolom
        # doc.close()
        doc = fitz.open(matching_pdf)
        pages_text = []

        for page in doc:
            pages_text.append(page.get_text("text", sort=True))

        doc.close()

        full_text = "\n".join(pages_text)
        words = full_text.split()

        if not words:
            print(f"⚠️ PDF {f_name} kosong atau tidak terbaca sebagai teks digital!")
            continue

        # Buat chunk per 1500 kata
        context_chunks = [" ".join(words[i : i+1500]) for i in range(0, len(words), 1500)]
        print(f"    ✅ Berhasil ekstraksi {len(words)} kata ke dalam {len(context_chunks)} chunk.")

    except Exception as e:
        print(f"❌ Error saat memproses PDF {f_name}: {e}")
        continue

    # Loop Soal yang tertunda
    for i, item in pending_tasks:
        print(f"    📍 Soal {i+1}/{len(qna_data)}", end=" ", flush=True)

        q_en = translate_text(item['Question'])
        a_en = translate_text(item['Answer'])

        best_score = 0
        best_fb = ""

        for idx, chunk in enumerate(context_chunks):
            print(".", end="", flush=True)

            instruction = f"""You are an expert in NLP evaluation metrics, specially trained to detect hallucinations in responses provided by language models. Evaluate whether the following answer contains hallucinations based on the given document context. PART {idx+1}.
            CONTEXT: {chunk}
            QUESTION: {q_en}
           Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context."""

            prompt = ABSOLUTE_PROMPT.format(
                instruction=instruction,
                response=a_en,
                reference_answer="",
                rubric=score_rubric
            )

            feedback, score = hf_judge(prompt)
            if score > best_score:
                best_score = score
                best_fb = f"(Chunk {idx+1}) {feedback}"

            if best_score == 5: break

        # Simpan ke CSV secara Real-time 
        with open(output_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                folder_name, p_range, i + 1, item['Question'], q_en,
                item['Answer'], a_en, best_score, convert_score(best_score),
                best_fb.replace('\n', ' '), best_score < 3, f_name
            ])
        print(f" Done! [Score: {best_score}]")

    torch.cuda.empty_cache()
    gc.collect()

print("\n🎉 SEMUA PROSES SELESAI!")

## **IDN-OCR**

In [ ]:
## versi dual GPU (running di kaggle)
import json
import pandas as pd
import pytesseract
import torch
import gc
import os
import re
import glob
import csv
from pdf2image import convert_from_path
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

# =========================================================
# 1. CONFIGURATION & PATHS
# =========================================================
JSON_DIR = '/kaggle/input/datasets/halizaapalwan/data-json/D8-Berdamai Dengan Diri Sendiri'
PDF_DIR = '/kaggle/input/datasets/halizaapalwan/dokumen/D8-Berdamai Dengan Diri Sendiri'
OUTPUT_BASE_PATH = '/kaggle/working/evaluation_results_v4/'
os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

# =========================================================
# 2. MODELS LOADING (Dual GPU Allocation)
# =========================================================
torch.cuda.empty_cache()
gc.collect()

# GPU 1 (cuda:1) -> NLLB Translation
print("📦 Loading NLLB (Translation) on GPU 1...")
trans_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
trans_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
).to("cuda:1").half() 

# GPU 0 (cuda:0) -> Prometheus 7B
print("🤖 Loading Prometheus (Judge) on GPU 0...")
eval_tokenizer = AutoTokenizer.from_pretrained("prometheus-eval/prometheus-7b-v2.0")
eval_model = AutoModelForCausalLM.from_pretrained(
    "prometheus-eval/prometheus-7b-v2.0",
    device_map={"": "cuda:0"}, 
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
)

# =========================================================
# 3. HELPERS
# =========================================================
def extract_page_range(filename):
    match = re.search(r'(\d+)-(\d+)', filename)
    return f"{match.group(1)}-{match.group(2)}" if match else None

def convert_score(score):
    return (score - 1) / 4.0

def translate_text(text):
    if not text or text.strip() == "": return ""
    trans_tokenizer.src_lang = "ind_Latn"
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda:1")
    with torch.no_grad():
        res = trans_model.generate(
            **inputs, 
            forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=512
        )
    return trans_tokenizer.decode(res[0], skip_special_tokens=True)

def hf_judge(prompt):
    safe_limit = 3584 
    inputs = eval_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=safe_limit).to("cuda:0")
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    decoded = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    match = re.search(r'\[RESULT\]\s*(\d)', decoded)
    score = int(match.group(1)) if match else 1
    feedback = decoded.split("[RESULT]")[0].replace("[FEEDBACK]", "").strip()
    return feedback, score

# =========================================================
# 4. RUBRIC SETUP 
# =========================================================
rubric_data = {
    "criteria": "Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context.",
    "score1_description": "The answer contains severe hallucinations with information that completely contradicts or is entirely absent from the document context.",
    "score2_description": "The answer contains significant hallucinations with multiple pieces of information that are inconsistent or add details not present in the context.",
    "score3_description": "The answer contains moderate hallucinations with some information that doesn't align with the context or includes inappropriate generalizations.",
    "score4_description": "The answer is mostly consistent with the context but contains minor hallucinations or slightly inaccurate interpretations.",
    "score5_description": "The answer is completely faithful to the context with no hallucinations, and all information is accurately derived from the document."
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# =========================================================
# 5. MAIN PROCESSING LOOP (Resume Logic)
# =========================================================
json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))

folder_name = os.path.basename(JSON_DIR)
output_csv = os.path.join(OUTPUT_BASE_PATH, f"{folder_name}_results.csv")

# LOGIC RESUME: Ambil daftar file yang sudah selesai diproses
processed_files = set()
if os.path.exists(output_csv):
    try:
        df_check = pd.read_csv(output_csv)
        if 'source_file' in df_check.columns:
            processed_files = set(df_check['source_file'].unique())
            print(f"🔄 Resume: {len(processed_files)} file terdeteksi sudah selesai.")
    except Exception as e:
        print(f"⚠️ Warning: Gagal membaca file lama, mulai dari awal. Error: {e}")

# Inisialisasi CSV jika benar-benar baru
if not os.path.exists(output_csv):
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'folder', 'page_range', 'question_id', 'question_id_original',
            'question_en', 'answer_id_original', 'answer_en',
            'score_5', 'score_01', 'feedback', 'is_hallucinated', 'source_file'
        ])

for j_path in tqdm(json_files, desc="🚀 Total Progress"):
    file_name = os.path.basename(j_path)
    
    # SKIP jika sudah ada di processed_files
    if file_name in processed_files:
        continue

    p_range = extract_page_range(file_name)
    matching_pdf = next((p for p in pdf_files if extract_page_range(os.path.basename(p)) == p_range), None)
    
    if not matching_pdf:
        continue

    # 1. OCR (CPU)
    images = convert_from_path(matching_pdf, dpi=150)
    raw_context = "\n".join([pytesseract.image_to_string(img, lang='ind') for img in images])
    
    # 2. Translate Context (GPU 1)
    raw_chunks = [raw_context[i:i+1500] for i in range(0, len(raw_context), 1500)]
    context_en = " ".join([translate_text(c) for c in raw_chunks])

    # 3. Chunking for Prometheus
    words = context_en.split()
    context_chunks = [" ".join(words[i : i+1500]) for i in range(0, len(words), 1500)]

    with open(j_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)

    # 4. Loop QnA
    for i, item in enumerate(qna_data):
        q_en = translate_text(item['Question'])
        a_en = translate_text(item['Answer'])
        
        best_score = 0
        best_fb = ""

        # Evaluasi terhadap context chunks (GPU 0)
        for idx, chunk in enumerate(context_chunks):
            instruction = f"""You are an expert in NLP evaluation metrics, specially trained to detect hallucinations in responses provided by language models. Evaluate whether the following answer contains hallucinations based on the given document context. PART {idx+1}.

            DOCUMENT CONTEXT:
            {chunk}

            QUESTION:
            {q_en}
           Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context."""
        

            prompt = ABSOLUTE_PROMPT.format(
                instruction=instruction,
                response=a_en,
                reference_answer="",
                rubric=score_rubric
            )
            
            feedback, score = hf_judge(prompt)
            
            if score > best_score:
                best_score = score
                best_fb = f"(Chunk {idx+1}) {feedback}"
            
            if best_score == 5: break 

        # Append hasil ke CSV
        with open(output_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                folder_name, p_range, i + 1, item['Question'], q_en,
                item['Answer'], a_en, best_score, convert_score(best_score),
                best_fb.replace('\n', ' '), best_score < 3, file_name
            ])

    torch.cuda.empty_cache()
    gc.collect()

print(f"\n🎉 Selesai! Hasil akhir tersimpan di: {output_csv}")

## **IDN**

In [ ]:
import json
import pandas as pd
import PyPDF2  
import torch
import gc
import os
import re
import glob
import csv
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

# =========================================================
# 1. CONFIGURATION & PATHS
# =========================================================
JSON_DIR = '/content/drive/MyDrive/KP/dataset/idn/D17-Dampak Gadget'
PDF_DIR = '/content/drive/MyDrive/KP/dokumen/idn/D17-Dampak Gadget'
OUTPUT_BASE_PATH = '/content/drive/MyDrive/KP/IDN-D17'
output_csv = os.path.join(OUTPUT_BASE_PATH, 'D17-Dampak Gadget_resultsH.csv')
os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

# =========================================================
# 2. MODELS LOADING
# =========================================================
print("📦 Loading NLLB (Translation) on CPU...")
trans_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
trans_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M").to("cpu")

print("🤖 Loading Prometheus (Judge) in 4-bit...")
eval_tokenizer = AutoTokenizer.from_pretrained("prometheus-eval/prometheus-7b-v2.0")
eval_model = AutoModelForCausalLM.from_pretrained(
    "prometheus-eval/prometheus-7b-v2.0",
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
)

# =========================================================
# 3. HELPERS
# =========================================================
def extract_page_range(filename):
    match = re.search(r'(\d+)-(\d+)', filename)
    return f"{match.group(1)}-{match.group(2)}" if match else None

def convert_score(score):
    return (score - 1) / 4.0

def translate_text(text):
    if not text or text.strip() == "":
        return ""
    trans_tokenizer.src_lang = "ind_Latn"
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cpu")
    with torch.no_grad():
        res = trans_model.generate(
            **inputs,
            forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=512
        )
    return trans_tokenizer.decode(res[0], skip_special_tokens=True)

def hf_judge(prompt):
    safe_limit = 3584
    inputs = eval_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=safe_limit).to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    decoded = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    match = re.search(r'\[RESULT\]\s*(\d)', decoded)
    score = int(match.group(1)) if match else 1
    feedback = decoded.split("[RESULT]")[0].replace("[FEEDBACK]", "").strip()
    return feedback, score

# =========================================================
# 4. RUBRIC SETUP
# =========================================================
rubric_data = {
    "criteria": "Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context.",
    "score1_description": "The answer contains severe hallucinations with information that completely contradicts or is entirely absent from the document context.",
    "score2_description": "The answer contains significant hallucinations with multiple pieces of information that are inconsistent or add details not present in the context.",
    "score3_description": "The answer contains moderate hallucinations with some information that doesn't align with the context or includes inappropriate generalizations.",
    "score4_description": "The answer is mostly consistent with the context but contains minor hallucinations or slightly inaccurate interpretations.",
    "score5_description": "The answer is completely faithful to the context with no hallucinations, and all information is accurately derived from the document."
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

# =========================================================
# 5. RESUME LOGIC: Load existing progress
# =========================================================
processed_set = set() 
if os.path.exists(output_csv):
    print(f"🔄 Menemukan file output lama. Membaca progres...")
    try:
        existing_df = pd.read_csv(output_csv)
        for _, row in existing_df.iterrows():
            unique_id = f"{row['source_file']}_{row['question_id']}"
            processed_set.add(unique_id)
        print(f"✅ Berhasil memuat {len(processed_set)} baris yang sudah diproses.")
    except Exception as e:
        print(f"⚠️ Gagal membaca file lama (mungkin kosong atau korup): {e}")

# Initialize CSV Header if not exists
if not os.path.exists(output_csv):
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'folder', 'page_range', 'question_id', 'question_id_original',
            'question_en', 'answer_id_original', 'answer_en',
            'score_5', 'score_01', 'feedback', 'is_hallucinated', 'source_file'
        ])

# =========================================================
# 6. MAIN PROCESSING LOOP
# =========================================================
json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
pdf_files = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
folder_name = os.path.basename(JSON_DIR)

for j_path in tqdm(json_files, desc="🚀 Files Progress"):
    j_filename = os.path.basename(j_path)
    p_range = extract_page_range(j_filename)
    matching_pdf = next((p for p in pdf_files if extract_page_range(os.path.basename(p)) == p_range), None)

    if not matching_pdf:
        continue

    with open(j_path, 'r', encoding='utf-8') as f:
        qna_data = json.load(f)

    needed_questions_idx = [i for i, _ in enumerate(qna_data) if f"{j_filename}_{i+1}" not in processed_set]
    total_needed = len(needed_questions_idx)

    if total_needed == 0:
        continue

    print(f"--- 📖 Memproses {total_needed} soal tersisa di: {j_filename} ---")
    
    raw_context = ""
    try:
        with open(matching_pdf, 'rb') as pdf_file:
            reader = PyPDF2.PdfReader(pdf_file)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    raw_context += text + "\n"
    except Exception as e:
        print(f"⚠️ Gagal membaca PDF {matching_pdf}: {e}")
        continue

    raw_chunks = [raw_context[i:i+1500] for i in range(0, len(raw_context), 1500)]
    context_en = " ".join([translate_text(c) for c in raw_chunks])

    words = context_en.split()
    context_chunks = [" ".join(words[i:i+1500]) for i in range(0, len(words), 1500)]

    print(f"    ✅ Berhasil ekstraksi {len(words)} kata ke dalam {len(context_chunks)} chunk.")

    for i, item in enumerate(qna_data):
        q_id = i + 1
        current_id = f"{j_filename}_{q_id}"

        if current_id in processed_set:
            continue

        print(f"    📍 Soal {q_id}/{len(qna_data)} ...... ", end="", flush=True)

        q_en = translate_text(item['Question'])
        a_en = translate_text(item['Answer'])

        best_score = 0
        best_fb = ""

        for idx, chunk in enumerate(context_chunks):
            instruction = f"""You are an expert in NLP evaluation metrics, specially trained to detect hallucinations in responses provided by language models. Evaluate whether the following answer contains hallucinations based on the given document context. PART {idx+1}.
            CONTEXT: {chunk}
            QUESTION: {q_en}
           Evaluate whether the AI-generated response contains any hallucinated or inaccurate information that deviates from the provided document context."""

            prompt = ABSOLUTE_PROMPT.format(
                instruction=instruction,
                response=a_en,
                reference_answer="",
                rubric=score_rubric
            )

            feedback, score = hf_judge(prompt)

            if score > best_score:
                best_score = score
                best_fb = f"(Chunk {idx+1}) {feedback}"

            if best_score == 5:
                break

        print(f"Done! [Score: {best_score}]")

        with open(output_csv, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                folder_name, p_range, q_id, item['Question'],
                q_en, item['Answer'], a_en,
                best_score, convert_score(best_score),
                best_fb.replace('\n', ' '), best_score < 3, j_filename
            ])

    torch.cuda.empty_cache()
    gc.collect()

print("\n🎉 SEMUA SELESAI! CSV final tersimpan di Working Directory.")